# 每日新闻摘要器

## 练习目标（理念）

做一个面向印度报纸《The Hindu》首页的**每日新闻摘要**小工具：

- **输入**：报纸首页 URL（默认 `https://www.thehindu.com/`）
- **中间步骤**：简易网页抓取（BeautifulSoup 清洗正文）
- **输出**：用本地 **llama3.2** 把重要标题整理成 **HTML**（多条标题 + 简述）

这是第 1 周 **Day 2** 作业思路的社区实践：抓取 + Prompt + 本地模型，不依赖云端 OpenAI Key。

## 和本课概念的关系

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| 网页抓取 | `requests` + `BeautifulSoup`，去掉 script/style 等噪声 |
| Chat Completions | `ollama.chat.completions.create(...)` |
| `messages`（system / user） | system 定「记者助手 + HTML 版式」，user 放「至少 8 条标题」+ 网页正文 |
| 本地 Ollama | `base_url=http://localhost:11434/v1`，模型 `llama3.2` |
| 笔记本展示 | `IPython.display.HTML` 直接渲染模型返回的 HTML |

## 怎么跑

1. 启动 Ollama，并确保已 `ollama pull llama3.2`
2. 从上到下依次运行单元格（Shift+Enter）
3. 先跑「连通性检查」格，确认 `localhost:11434` 有响应
4. 最后一格会对 The Hindu 首页做摘要；抓取与推理可能较慢，属正常


In [1]:
# ========== 导入：把后面要用的工具箱搬进来 ==========

# 导入 requests：HTTP 抓取网页，以及探测本地 Ollama 是否在听端口
import requests
# 从 openai 导入 OpenAI 客户端：通过兼容接口调用本地 Ollama
from openai import OpenAI
# 从 IPython.display 导入 HTML：把模型返回的 HTML 字符串直接渲染在笔记本里
from IPython.display import HTML


### 抓取代码

下面定义 `headers` 与 `fetch_website_info(url)`：伪装浏览器访问目标站，解析 HTML，去掉脚本/样式等噪声，返回「标题 + 正文」并截断到 2000 字符，避免把过长上下文塞进模型。


In [9]:
# ========== 抓取：URL → 清洗后的 title + 正文（截断）==========

# 从 bs4 导入 BeautifulSoup：把字节 HTML 变成可查询的文档树
from bs4 import BeautifulSoup

# 浏览器 User-Agent：降低被新闻站按默认爬虫 UA 拒绝的概率
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36"}

def fetch_website_info(url):
    # 抓取网页正文，截断到 2000 字符，控制送进模型的上下文长度
    # GET 目标 URL；带上伪装浏览器的 headers
    response = requests.get(url, headers=headers)
    # 解析 HTML（参数间距与原代码保持一致，不改逻辑）
    soup = BeautifulSoup(response.content,"html.parser")
    # 取页面 <title>；没有则用占位英文（保留原字符串）
    title = soup.title.string if soup.title else "No title found"

    if soup.body:
        # 去掉脚本、样式、图片、表单等无关节点，减少噪声
        for irrelevant in soup.body(["script", "style", "img", "input"]):
            irrelevant.decompose()
        # 抽出纯文本：换行分隔并去掉首尾空白
        text = soup.body.get_text(separator="\n", strip=True)
    else:
        # 无 body 时正文为空
        text = ""
    # 标题与正文拼接后截断到 2000 字符返回
    return (title + "\n\n" + text)[:2_000]


In [10]:
# ========== 检查本地 Ollama 是否在运行 ==========

# 对默认端口发 GET；能返回内容通常说明 Ollama 进程在听 11434
# 若连接失败，请先在终端启动 ollama serve / 桌面版 Ollama
requests.get("http://localhost:11434").content


b'Ollama is running'

In [11]:
# ========== 通过 OpenAI 兼容接口连接本地 Ollama ==========

# Ollama 的 OpenAI 兼容基址（注意带 /v1）
OLLAMA_BASE_URL = 'http://localhost:11434/v1'
# api_key 为 SDK 必填占位；本地 Ollama 会忽略真实鉴权
ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key="ollama")


In [12]:
# ========== 提示词：system 定「记者 + HTML 版式」，user 要求至少 8 条标题 ==========
# 发给模型的英文 prompt 保留不译（改译会改变输出风格/行为）

# 系统提示：要求以 HTML 输出多条新闻标题与简述，跳过广告与报纸自我介绍
system_prompt = """You are an assistant journalist. You are tasked with summarizing any important news headline in the current day's newspaper. Respond in html.  Summarize the info in separate headings and paragraphs. Heading would be the headline and a small description of it as the paragraph. Skip any advertisements or description about the newpaper or anything that might not be news"""

# 用户提示前缀：说明这是 The Hindu，并要求至少 8 条标题；后面会拼接网页正文
user_prompt = """Here is the website information for the newspaper, the hindu. Include atleast 8 headlines"""


In [13]:
# ========== headline_summarizer：抓取 → 组 messages → 调模型 → 返回 HTML 展示对象 ==========

def headline_summarizer(url):
    # 抓取网页 → 拼消息 → 调用模型 → 以 HTML 展示
    # 先抓取并清洗目标报纸首页
    summary = fetch_website_info(url)
    # system + user（user = 固定前缀 + 网页正文）
    messages = [{"role":"system", "content":system_prompt},
            {"role":"user", "content":user_prompt + summary}]
    # 非流式调用本地 llama3.2；返回完整 HTML 字符串
    response = ollama.chat.completions.create(model="llama3.2", messages=messages)
    # 包装成 IPython HTML，笔记本会按 HTML 渲染而不是转义成纯文本
    return HTML(response.choices[0].message.content)


In [ ]:
# ========== 对 The Hindu 首页做摘要（端到端）==========

# 默认目标：印度《The Hindu》首页；可改成其它公开新闻站做试验
url = "https://www.thehindu.com/"
# 调用封装函数：抓取 + 推理；输出为 HTML 展示对象
headline_summarizer(url)
